# Teste do Modelo 5: Datadog Toto

In [1]:
import pandas as pd
import torch
import os

# --- 1. Configurações ---
DATA_DIR = "../../data"
HORIZONTE_PREVISAO = 14
N_DIAS_HISTORICO = 100

# --- 2. Carregar Dados ---
print("Carregando dados...")
hist_path = os.path.join(DATA_DIR, "hist.parquet")
df_hist = pd.read_parquet(hist_path)
print("Dados históricos carregados.")

# --- 3. Definir dispositivo ---
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"\nUsando dispositivo: {device}")

Carregando dados...
Dados históricos carregados.

Usando dispositivo: cpu


In [2]:
# ==============================================================================
# TESTE 5: Datadog Toto
# ==============================================================================
# API baseada na documentação oficial: https://github.com/DataDog/toto
# ==============================================================================

try:
    from toto.model.toto import Toto
    from toto.inference.forecaster import TotoForecaster
    from toto.data.util.dataset import MaskedTimeseries

    print("\n--- Testando Datadog Toto ---")

    # 1. Carregar o modelo pré-treinado
    toto = Toto.from_pretrained('Datadog/Toto-Open-Base-1.0')
    toto.to(device)
    forecaster = TotoForecaster(toto.model)

    # 2. Preparar dados
    # O modelo espera o formato (channels, time_steps)
    # Para nossa série univariada, 'channels' é 1.
    input_series = torch.tensor(df_hist['target'].values, dtype=torch.float32).reshape(1, N_DIAS_HISTORICO).to(device)
    
    # A API espera informações de timestamp, mas o modelo atual não as usa.
    # Criamos tensores de preenchimento como no exemplo oficial.
    inputs = MaskedTimeseries(
        series=input_series,
        padding_mask=torch.full_like(input_series, True, dtype=torch.bool),
        id_mask=torch.zeros_like(input_series),
        timestamp_seconds=torch.zeros_like(input_series),
        time_interval_seconds=torch.full((1,), 60*60*24).to(device), # Frequência diária em segundos
    )

    # 3. Rodar a previsão
    print(f"Rodando previsão para {HORIZONTE_PREVISAO} passos...")
    forecast = forecaster.forecast(
        inputs,
        prediction_length=HORIZONTE_PREVISAO,
        num_samples=256, # Como recomendado na documentação
        samples_per_batch=256,
    )

    # Acessar o resultado (mediana das amostras)
    median_prediction = forecast.median.cpu().numpy().flatten()

    print("Previsão (primeiros 5 valores):", median_prediction.round(2)[:5])
    print("Teste do Toto concluído.\n")

except ImportError:
    print("Toto-ts não instalado. Pulando teste.")
except Exception as e:
    print(f"Erro ao rodar Toto: {e}")

/app/time_series_models/notebooks/05_toto/.venv/lib/python3.12/site-packages/einops/einops.py:847: SyntaxWarning: invalid escape sequence '\s'
  \sum_{c, d, g} x[a, b, c] * y[c, b, d] * z[a, g, k]


/app/time_series_models/notebooks/05_toto/.venv/lib/python3.12/site-packages/lightning/fabric/__init__.py:40: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.


/app/time_series_models/notebooks/05_toto/.venv/lib/python3.12/site-packages/gluonts/json.py:102: UserWarning: Using `json`-module for json-handling. Consider installing one of `orjson`, `ujson` to speed up serialization and deserialization.
  warnings.warn(



--- Testando Datadog Toto ---


config.json:   0%|          | 0.00/582 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/605M [00:00<?, ?B/s]

Rodando previsão para 14 passos...


Previsão (primeiros 5 valores): [115.   108.09  95.41  88.4   89.67]
Teste do Toto concluído.

